# Run Infomap on HPC

Infomap trials are independent work units. That makes them a good fit for HPC: run many trials on one node with `--parallel-trials`, or split trials across a scheduler job array with `--trial-offset` and merge the shard results afterwards.

This notebook uses Python as orchestration glue. The expensive work runs in the native `Infomap` command-line binary. The final merge uses the Python helper `python -m infomap.merge`, which reads shard JSON files and copies the winning tree without rerunning Infomap.

## Build for your cluster

Start by checking which compiler, OpenMP runtime, and flags Infomap will use:

```bash
make doctor
```

A portable OpenMP build is the usual starting point:

```bash
make build-native OPENMP=1
```

For a homogeneous partition where build and run nodes have the same CPU type, you can opt into node-local tuning:

```bash
make build-native OPENMP=1 NATIVE_ARCH=1
```

`NATIVE_ARCH=1` enables non-portable native tuning such as `-march=native`, link-time optimization, and loop unrolling. Build it on the same kind of node where you will run it. If OpenMP is hard to configure on your cluster, build without it and scale with job arrays instead:

```bash
make build-native OPENMP=0
```

In [ ]:
from __future__ import annotations

import json
import math
import os
import shlex
import subprocess
import sys
import tempfile
from pathlib import Path

from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "main.cpp").exists() and (path / "examples" / "notebooks").exists():
            return path
    raise RuntimeError("Run this notebook from an Infomap source checkout.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
INFOMAP = REPO_ROOT / ("Infomap.exe" if os.name == "nt" else "Infomap")
NETWORK = REPO_ROOT / "examples" / "networks" / "ninetriangles.net"
_tmp = tempfile.TemporaryDirectory(prefix="infomap-hpc-")
WORK = Path(_tmp.name)
RUN_LIVE = INFOMAP.exists()


def command_text(cmd: list[str]) -> str:
    return shlex.join(str(part) for part in cmd)


def run_cli(cmd: list[str], *, cwd: Path | None = None, env: dict[str, str] | None = None):
    merged_env = os.environ.copy()
    if env:
        merged_env.update(env)
    print("$", command_text(cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=merged_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}")
    return result


if RUN_LIVE:
    display(Markdown(f"Using native Infomap binary: `{INFOMAP}`"))
else:
    display(Markdown(
        "Native `Infomap` binary not found. Run `make build-native` from the "
        "repository root, then rerun this notebook to execute the live HPC recipes."
    ))

print(f"work directory: {WORK}")

## Single-node recipe

Use `--parallel-trials` when one allocated node has enough cores and memory for the whole run. `--num-threads auto` picks up scheduler limits such as `SLURM_CPUS_PER_TASK`, so the process does not silently use every core on a shared node.

In [ ]:
single_dir = WORK / "single-node"
single_dir.mkdir(exist_ok=True)

single_cmd = [
    str(INFOMAP),
    str(NETWORK),
    str(single_dir),
    "--num-trials", "6",
    "--parallel-trials",
    "--num-threads", "auto",
    "--seed", "123",
    "--timing-json", str(single_dir / "timing.json"),
    "--summary-json", str(single_dir / "summary.json"),
    "--manifest-json", str(single_dir / "manifest.json"),
    "--silent",
]

if RUN_LIVE:
    run_cli(single_cmd, env={"SLURM_CPUS_PER_TASK": "2"})
    timing = json.loads((single_dir / "timing.json").read_text())
    summary = json.loads((single_dir / "summary.json").read_text())
    manifest = json.loads((single_dir / "manifest.json").read_text())

    print("thread source:", timing["thread_source"])
    print("threads used:", timing["threads_used"])
    print("trial codelengths:", summary["trial_codelengths"])
    print("config fingerprint:", manifest["config_fingerprint"])
else:
    display(Markdown("```bash\n" + command_text(single_cmd) + "\n```"))

In [ ]:
if RUN_LIVE:
    try:
        import matplotlib.pyplot as plt

        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(range(len(summary["trial_codelengths"])), summary["trial_codelengths"], marker="o")
        ax.set_xlabel("Trial")
        ax.set_ylabel("Codelength")
        ax.set_title("Independent trial results")
        ax.grid(True, alpha=0.3)
        plt.show()
    except ImportError:
        print("matplotlib is not installed; skipping plot.")

## Distributed trials on one machine

The job-array version uses the same commands. This notebook simulates two array tasks locally. Each shard runs a disjoint range of global trial indices and writes one `--trial-results` JSON file.

The important rule is that the reference run also uses sharding mode. A plain serial `--num-trials N` run uses legacy continuous RNG behavior and is not the right comparison for deterministic sharding.

In [ ]:
def trial_vector(path: Path) -> list[float]:
    data = json.loads(path.read_text())
    return [float(t["codelength"]) for t in sorted(data["trials"], key=lambda t: int(t["trial"]))]


def shard_command(out_dir: Path, *, num_trials: int, trial_offset: int, result_name: str) -> list[str]:
    out_dir.mkdir(parents=True, exist_ok=True)
    return [
        str(INFOMAP),
        str(NETWORK),
        str(out_dir),
        "--num-trials", str(num_trials),
        "--trial-offset", str(trial_offset),
        "--seed", "123",
        "--num-threads", "1",
        "--trial-results", str(out_dir / result_name),
        "--silent",
    ]


ref_dir = WORK / "reference"
shard0_dir = WORK / "shard-0"
shard1_dir = WORK / "shard-1"

ref_cmd = shard_command(ref_dir, num_trials=5, trial_offset=0, result_name="reference.json")
shard0_cmd = shard_command(shard0_dir, num_trials=2, trial_offset=0, result_name="results_0.json")
shard1_cmd = shard_command(shard1_dir, num_trials=3, trial_offset=2, result_name="results_1.json")

if RUN_LIVE:
    run_cli(ref_cmd)
    run_cli(shard0_cmd)
    run_cli(shard1_cmd)

    reference_vector = trial_vector(ref_dir / "reference.json")
    shard_vector = trial_vector(shard0_dir / "results_0.json") + trial_vector(shard1_dir / "results_1.json")
    assert reference_vector == shard_vector
    print("reference vector:", reference_vector)
    print("shard vector:    ", shard_vector)
else:
    display(Markdown("```bash\n" + "\n".join(command_text(cmd) for cmd in [ref_cmd, shard0_cmd, shard1_cmd]) + "\n```"))

## Merge shard results with Python

Merging is intentionally a Python post-processing step. It is cheap compared with the Infomap runs: it reads shard JSON, verifies matching fingerprints, selects the lowest-codelength global trial, and writes `tree` / `clu` output from the winning tree.

In [ ]:
merge_dir = WORK / "merge"
merge_dir.mkdir(exist_ok=True)
merge_cmd = [
    sys.executable,
    "-m", "infomap.merge",
    str(shard0_dir / "results_0.json"),
    str(shard1_dir / "results_1.json"),
    "--out-name", str(merge_dir / "final"),
    "--output", "tree,clu",
    "--require-complete-trials",
]

if RUN_LIVE:
    run_cli(merge_cmd)
    assert (merge_dir / "final.tree").exists()
    assert (merge_dir / "final.clu").exists()

    shard_files = [shard0_dir / "results_0.json", shard1_dir / "results_1.json"]
    shards = [json.loads(path.read_text()) for path in shard_files]
    trials = [trial for shard in shards for trial in shard["trials"]]
    winner = min(trials, key=lambda t: (float(t["codelength"]), int(t["trial"])))
    assert math.isclose(float(winner["codelength"]), min(reference_vector))
    print("winning global trial:", winner["trial"])
    print("winning codelength:", winner["codelength"])
else:
    display(Markdown("```bash\n" + command_text(merge_cmd) + "\n```"))

Programmatic merge uses the same implementation:

```python
from infomap.merge import merge_trial_results

summary = merge_trial_results(
    ["results_*.json"],
    out_name="final",
    formats=("tree", "clu"),
    require_complete=True,
)
```

## SLURM recipe

The array job runs the native binary. `--num-trials` is the per-shard count. `--trial-offset` maps each array task to a global trial range.

```bash
#!/usr/bin/env bash
#SBATCH --job-name=infomap-shards
#SBATCH --array=0-3
#SBATCH --cpus-per-task=8
#SBATCH --time=02:00:00

set -euo pipefail

INFOMAP=/path/to/Infomap
NETWORK=/path/to/graph.net
OUT=/path/to/out
TRIALS_PER_SHARD=25
OFFSET=$((SLURM_ARRAY_TASK_ID * TRIALS_PER_SHARD))

mkdir -p "$OUT/shards/$SLURM_ARRAY_TASK_ID"

srun "$INFOMAP" "$NETWORK" "$OUT/shards/$SLURM_ARRAY_TASK_ID" \
  --num-trials "$TRIALS_PER_SHARD" \
  --trial-offset "$OFFSET" \
  --seed 123 \
  --num-threads auto \
  --trial-results "$OUT/results_${SLURM_ARRAY_TASK_ID}.json" \
  --no-final-output \
  --silent
```

After the array completes, run the merge as a small Python post-processing job:

```bash
python -m infomap.merge '/path/to/out/results_*.json' \
  --out-name /path/to/out/final \
  --output tree,clu \
  --require-complete-trials
```

## Before scaling

- Use the same input, seed, and algorithm options for all shards.
- Treat `--num-trials` as the per-shard count.
- Use non-overlapping `--trial-offset` ranges.
- Keep the reference comparison in sharding mode.
- Use `--require-complete-trials` when gaps should fail the merge.
- Check exit codes in batch scripts. Input and output failures should not look like successful runs.
- Remember that merge can write `tree` and `clu`; link-bearing formats need a full Infomap run.